<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-HUYTRUONG/AIO/blob/keep-track/M3/ML-Base/Tree-based-Algorithms/decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Decision Tree**

Mục tiêu tài liệu:
- Ta có thể giải thích cách một **decision tree** đưa ra dự đoán.
- Phân biệt **Gini** và **Entropy**, huấn luyện mô hình bằng.
- Huấn luyện mô hình bằng `scikit-learn`.
- Đọc **ma trận nhầm lẫn (Confusion Matrix)** và nhận ra những dấu hiệu cơ bản của hiện tượng overfitting.

- Hãy hình dung ta cần dự đoán một chiếc cân sẽ nghiêng sang trái, nghiêng sang phải hoặc là sẽ giữa thăng bằng. Một cách tự nhiên là lần lượt đưa ra những câu hỏi: "Bên trái có nặng hơn không", "vật được đặt xa tâm cân hơn không", rồi từ đó có các câu trả lời dẫn đến kết luận. **Decision Tree** học đúng theo tinh thần đó. Khác biệt nằm ở chỗ các câu hỏi không do con người viết sẵn mà chúng được chọn từ dữ liệu.

- Tài liệu sử dụng bộ dữ liệu [Balance Scale của UCI ](https://archive.ics.uci.edu/dataset/12/balance+scale) để minh họa. Đây là ví một ví dụ nhỏ, đủ trực quan để quan sát toàn bộ quy trình nhưng vẫn bộc lộ những vấn đề thường gặp trong thực tế:
  - **Lớp mất cân bằng (Imbalanced Class problem)**: Hiện tượng số lượng samples giữa các classes trong dataset có sự chênh lệch rất lớn. Khi một hoặc vài lớp chiếm phần lớn data trong khi các lớp còn lại chỉ xuất hiện với tỷ lệ rất nhỏ $\rightarrow$ **Đánh giá sai hiệu năng bằng Accuracy**.
    - **Lớp R (Lệch phải)**: 288 mẫu (46.08%)
    - **Lớp L (Lệch trái)**: 288 mẫu (46.08%)
    - **Lớp B (Cân bằng)**: 49 mẫu (chỉ chiếm 7.84%)

    $\Rightarrow$ Lớp B chính là lớp hiếm/mất cân bằng so với hai lớp L và R

  - **Cây quá sâu** và cách diễn giải một con số **accuracy**.

## **1. Cây quyết định nhìn từ trực giác**

### **1.1. Các thành phần của cây**


| Thành phần | Tên tiếng Anh | Vai trò |
| :--- | :--- | :--- |
| **Nút gốc** | Root node | Chứa toàn bộ dữ liệu trước phép chia đầu tiên. |
| **Nút trong** | Internal node | Đặt một câu hỏi về đặc trưng và ngưỡng chia. |
| **Nhánh** | Branch | Biểu diễn kết quả của câu hỏi ở nút cha. |
| **Nút lá** | Leaf node | Chứa dự đoán cuối cùng của mô hình. |


### **1.2. Một chuỗi điều kiện**

- **Decision Tree** là một thuật toán **học có giám sát (Supervised Learning)**, dùng được cho cả phân loại và hồi quy. Thay vì cố gắng mô tả toàn bộ dữ liệu bằng một công thức duy nhất, cây chia không gian đặc trưng thành nhiều vùng nhỏ. Mỗi lần chia tương ứng với một câu hỏi, chẳng hạn như: $$\text{left_weight} \le 2{,}5?$$

- Một **Sample** bắt đầu ở **root node**, đi theo **branch** phù hợp với câu trả lời và dừng lại ở một **leaf node**.

  - Với bài toán:
    - **classification**, **leaf node** sẽ trả về một **class label**.
    - **regression**, nó thường trả về giá trị trung bình của các mẫu huấn luyện nằm ở **leaf node** đó.
  
  - Bằng cách ghép nhiều phép chia lại với nhau, một đường đi từ **root** đến **leaf** chính là một quy tắc gồm nhiều điều kiện.
  - Cây càng sâu thì quy tắc càng chi tiết, đồng thời cũng là nguyên nhân khiến cây dễ bị **overfitting (học vẹt dữ liệu huấn luyện)**.

## **2. Cách chia node trong decision tree**

### **2.1. Từ một node thành các nodes con**


| Thuật ngữ tiếng Việt | Thuật ngữ tiếng Anh | Ý nghĩa trong Cây quyết định (Decision Tree) |
| :--- | :--- | :--- |
| **Mẫu / Đối tượng** | **Sample** | Một dòng/bản ghi dữ liệu cụ thể chạy qua cây để phân loại hoặc dự đoán. |
| **Đặc trưng** | **Feature** | Thuộc tính/biến dữ liệu dùng làm căn cứ đặt câu hỏi rẽ nhánh (ví dụ: `weight`, `distance`). |
| **Ngưỡng chia** | **Threshold** | Giá trị mốc được chọn để làm điều kiện tách nhánh (ví dụ: $x_1 \le t_1$). |
| **Phép chia** | **Split** | Thao tác tách dữ liệu tại một nút thành các nút con. |
| **Độ hỗn tạp** | **Impurity** | Mức độ không thuần khiết/trộn lẫn giữa các lớp dữ liệu tại một nút (đo bằng Gini hoặc Entropy). |
| **Nút gốc** | **Root node** | Nút đầu tiên ở đỉnh cây, chứa toàn bộ tập dữ liệu ban đầu. |
| **Nút trong** | **Internal node** | Nút đặt câu hỏi kiểm tra điều kiện để tiếp tục phân nhánh. |
| **Nút con / Nút cha** | **Child node / Parent node** | Nút được tách ra (Child) từ nút liền trước nó (Parent). |
| **Nút lá** | **Leaf node** | Nút dừng cuối cùng trên cây, trả về kết quả dự đoán (nhãn lớp hoặc giá trị). |
| **Nhánh** | **Branch** | Đường nối biểu diễn kết quả trả lời của câu hỏi (đúng/sai). |
| **Thuần khiết** | **Pure** | Trạng thái mà tất cả các mẫu trong nút đều thuộc về cùng một lớp (Impurity = 0). |
| **Đệ quy** | **Recursive** | Quy trình lặp lại cùng một logic chia nút cho các nút con tiếp theo. |
| **Điều kiện dừng** | **Stopping criteria** | Các giới hạn (như `max_depth`, `min_samples_leaf`) để dừng việc chia nhỏ cây. |
| **Tập huấn luyện** | **Training set** | Tập dữ liệu dùng để học và dựng nên cấu trúc cây. |
| **Bài toán phân loại** | **Classification** | Bài toán dự đoán nhãn danh mục/lớp (Class label). |
| **Bài toán hồi quy** | **Regression** | Bài toán dự đoán giá trị con số liên tục. |
| **Hiện tượng quá khớp** | **Overfitting** | Mô hình học thuộc lòng dữ liệu huấn luyện, dự đoán kém trên dữ liệu mới. |

- Giả sử một **node** đang chứa một **tập mẫu $S$ (sample set $S$)**. Thuật toán thử nhiều cặp gồm các **features** và **threshold**. Với mỗi ứng viên, dữ liệu được **split** thành hai tập $S_L$ và $S_R$. Chất lượng của **split** được đánh giá bằng **impurity** có trọng số:

$$I_{split} = \frac{\vert{}S_L\vert{}}{\vert{}S\vert{}} I(S_L) + \frac{\vert{}S_R\vert{}}{\vert{}S\vert{}} I(S_R)$$

- Trong đó $I(\cdot)$ có thể là **Gini impurity** hoặc **Entropy**. Một **split** tốt tạo ra các **child nodes** 'thuần khiết (**pure**)' hơn **parent node**, nghĩa là mỗi **child node** chủ yếu chứa **samples** của một **class**.

- Quy trình này được lặp lại theo kiểu đệ quy:
  1. Bắt đầu với **toàn bộ tập huấn luyện (training set)** tại **root node**.
  2. Tìm **split** làm giảm **impurity** nhiều nhất.
  3. Đưa **samples** sang hai **child nodes** theo điều kiện vừa chọn.
  4. Tiếp tục **split** cho đến khi gặp **điều kiện dừng (stopping criteria)**.
  5. Gán giá trị **dự đoán (prediction)** cho từng **leaf node**.

<p align="center">
  <img src="https://github.com/AIVIETNAM-AIO-HUYTRUONG/AIO/blob/keep-track/Images/1687b639-7a15-4fdd-bc6c-fe0ce40ec6a8.png?raw=1" alt="1687b639-7a15-4fdd-bc6c-fe0ce40ec6a8"/>
</p>

### **2.2. Gini impurity**

| Thuật ngữ tiếng Việt | Thuật ngữ tiếng Anh | Ý nghĩa / Giải thích |
| :--- | :--- | :--- |
| **Độ hỗn tạp Gini** | **Gini impurity** | Thước đo mức độ trộn lẫn/không thuần khiết của các lớp tại một nút . |
| **Lớp / Nhãn lớp** | **Class / Class label** | Danh mục phân loại của dữ liệu (ví dụ: Lớp L, Lớp R, Lớp B) . |
| **Nút thuần** | **Pure node** | Nút chứa 100% mẫu thuộc về duy nhất một lớp (Gini = 0) . |
| **Gán nhãn** | **Label assignment / Assign label** | Thao tác gán một nhãn danh mục cho mẫu dữ liệu . |
| **Khả năng bị gán sai** | **Misclassification rate / Probability** | Xác suất hoặc tỷ lệ một mẫu bị dự đoán/gán sai nhãn . |

- Với $K$ **classes** và $p_k$ là tỷ lệ **samples** thuộc **class** $k$ tại một **node**, **Gini impurity** được tính bởi:

$$Gini(S) = 1 - \sum_{k=1}^{K} p_k^2$$

- **Gini** bằng $0$ khi toàn bộ **samples** tại **node** thuộc cùng một **class (pure node)**. Giá trị càng lớn thì các **classes** càng trộn lẫn (hỗn tạp). Chẵng hạn, một **node** gồm **6 samples**, trong đó có **4 samples** thuộc  **class $L$** và $2$ samples thuộc **class $R$**, có:

$$Gini(S) = 1 - \left(\frac{4}{6}\right)^2 - \left(\frac{2}{6}\right)^2 = \frac{4}{9} \approx 0.444$$

- Có thể hiểu **Gini** theo một trục giác gần đúng: nếu gắn nhãn ngẫu nhiên theo đúng tỷ lệ lớp ở **node**, chỉ số này phẩn ánh xác xuất một **sample** bị gán sai.

<p align="center">
  <img src="https://github.com/AIVIETNAM-AIO-HUYTRUONG/AIO/blob/keep-track/Images/01402546-83b3-454d-9620-badd7dfdda6b.png?raw=1" alt="01402546-83b3-454d-9620-badd7dfdda6b"/>
</p>

### **2.3. Entropy và Information Gain**

| Thuật ngữ tiếng Việt | Thuật ngữ tiếng Anh | Ý nghĩa / Giải thích |
| :--- | :--- | :--- |
| **Độ hỗn loạn / Mức bất định** | **Entropy / Uncertainty** | Thước đo mức độ bất định hoặc ngẫu nhiên của các lớp dữ liệu tại một nút . |
| **Mức tăng thông tin** | **Information Gain (IG)** | Lượng bất định giảm đi sau khi thực hiện một phép chia nút (chênh lệch giữa entropy nút cha và tổng entropy có trọng số của các nút con) . |
| **Phân phối lớp** | **Class distribution** | Tỷ lệ xuất hiện của các lớp dữ liệu khác nhau tại một nút . |
| **Tối thiểu hóa / Tối đa hóa** | **Minimize / Maximize** | Mục tiêu giảm thiểu chỉ số xấu (Gini/Entropy) hoặc tối đa hóa chỉ số tốt (Information Gain) . |

- **Entropy** đo **mức bất định (uncertainty)** của phân phối **class**:

$$H(S) = -\sum_{k=1}^{K} p_k \log_2 p_k$$

- Với quy ước giới hạn $p_k \log_2 p_k = 0$ khi $p_k = 0$. Một **pure node** có **entropy** bằng $0$.
- Khi các **classes** xuất hiện với tỷ lệ gần nhau, **entropy** tăng lên. Nếu dùng **entropy**, chất lượng của **split** thường được viết dưới dạng **Information Gain**

$$IG(S) = H(S) - \frac{\vert{}S_L\vert{}}{\vert{}S\vert{}} H(S_L) - \frac{\vert{}S_R\vert{}}{\vert{}S\vert{}} H(S_R)$$

- **Information Gain** càng lớn thì **uncertainty** bị loại bỏ sau khi **split** càng nhiều. Nói ngắn gọn: **Gini impurity** và **Entropy** là những đại lượng ta muốn giảm (minimize).
- **Information Gain** là đại lượng ta muốn tăng (maximize)."

<p align="center">
  <img src="https://github.com/AIVIETNAM-AIO-HUYTRUONG/AIO/blob/keep-track/Images/402c8db8-4e59-446c-82a5-e390b21b7c55.png?raw=1" alt="402c8db8-4e59-446c-82a5-e390b21b7c55"/>
</p>

### **2.4. Điều kiện dừng và bài toán overfit**


| Thuật ngữ / Tham số | Tên tiếng Anh / Code | Ý nghĩa trong kiểm soát Cây quyết định |
| :--- | :--- | :--- |
| **Siêu tham số** | **Hyperparameter** | Các tham số được thiết lập trước khi huấn luyện để điều chỉnh và chống quá khớp (overfitting). |
| **Độ sâu tối đa** | `max_depth` | Giới hạn số tầng rẽ nhánh từ gốc đến lá để cây không mọc quá sâu[cite: 1]. |
| **Mẫu tối thiểu để chia** | `min_samples_split` | Một node phải có đủ số mẫu tối thiểu này mới được phép tiếp tục split[cite: 1]. |
| **Mẫu tối thiểu tại lá** | `min_samples_leaf` | Mỗi leaf node bắt buộc phải chứa ít nhất số mẫu này, giúp chặn các quy tắc quá chi tiết/học thuộc lòng[cite: 1]. |
| **Số lá tối đa** | `max_leaf_nodes` | Giới hạn tổng số nút lá trên toàn bộ cây[cite: 1]. |
| **Hệ số cắt tỉa** | `ccp_alpha` | Tham số loại bỏ bớt các nhánh dư thừa (Pruning) để tối ưu hóa độ phức tạp của cây[cite: 1]. |

- Nếu không bị giới hạn, cây có thể tiếp tục **split** cho đến khi các **leaf nodes** gần như **pure** hoàn toàn. **Accuracy** trên **tranining set** khi đó thường rất cao (gần như 100%), nhưng mô hình dễ bị **overfiting** và dự đoán kém trên **test set**.

- Một số **tham số (hyperparameters)** quan trọng:
    - `max_depth`: Độ sâu tối đa của cây (tree depth).
    - `min_samples_split`: Số **samples** tối thiểu tại một **node** để được phép tiếp tục **split**.  
    - `min_samples_leaf`: Số **samples** tối thiểu phải còn lại tại mỗi **leaf node**.  
    - `max_leaf_nodes`: Số **leaf nodes** tối đa của cây.  
    - `ccp_alpha`: Mức cắt tỉa cây (pruning) theo thuật toán cost-complexity pruning."